# Day 4 概念实验：第一周 Transformer 复习

本 notebook 将本周概念串成三组最小实验：**缩放为什么能避免 softmax 过度尖锐？位置编码如何打破自注意力对置换的等变性？序列变长时注意力矩阵为何迅速变贵？**

完整知识脉络、问答和延伸阅读见 `ima/第1周-Day4-第一周总复习.md`；本处只做可执行验证。

## 实验 1：为什么分数要除以 sqrt(d_k)？

随机 Q、K 的分量方差固定时，点积方差随维度增长。比较缩放前后的分数标准差与 softmax 熵；熵越低代表分布越接近“一票独大”。

In [ ]:
import numpy as np
rng = np.random.default_rng(2026)
def softmax(x):
    x = x - x.max(axis=-1, keepdims=True); e = np.exp(x); return e/e.sum(axis=-1, keepdims=True)
def entropy(p): return float((-p*np.log(p+1e-12)).sum(axis=-1).mean())
widths = np.array([4, 16, 64, 256])
raw_std, scaled_std, raw_entropy, scaled_entropy = [], [], [], []
for d in widths:
    Q = rng.normal(size=(4000,d)); K = rng.normal(size=(4000,d))
    dots = (Q*K).sum(axis=1)
    raw_std.append(dots.std()); scaled_std.append((dots/np.sqrt(d)).std())
    q = rng.normal(size=(1000,6,d)); k = rng.normal(size=(1000,d))
    logits = np.einsum("bnd,bd->bn", q, k)
    raw_entropy.append(entropy(softmax(logits))); scaled_entropy.append(entropy(softmax(logits/np.sqrt(d))))
print("d_k | 原始标准差 | 缩放后标准差 | 原始熵 | 缩放后熵")
for row in zip(widths,raw_std,scaled_std,raw_entropy,scaled_entropy): print(f"{row[0]:>3} | {row[1]:>10.2f} | {row[2]:>12.2f} | {row[3]:>6.2f} | {row[4]:>8.2f}")
assert max(scaled_std)-min(scaled_std) < 0.15
print("结论：除以 sqrt(d_k) 将点积的典型尺度拉回约 1。")

## 实验 2：缩放如何保留可分配的注意力？

图中比较不同键维度的平均 softmax 熵。未缩放时维度越大，分数越极端；缩放后注意力分布更稳定。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(widths, raw_entropy, "o-", label="未缩放"); ax.plot(widths, scaled_entropy, "o-", label="除以 sqrt(d_k)")
ax.set_xscale("log", base=2); ax.set_xticks(widths, [str(x) for x in widths]); ax.set_ylim(0, np.log(6)+0.1)
ax.set_xlabel("键维度 d_k"); ax.set_ylabel("平均 softmax 熵（6 个候选）"); ax.set_title("缩放让注意力尖锐度更稳定"); ax.grid(alpha=.25); ax.legend()
plt.tight_layout(); plt.show()

## 实验 3：位置编码怎样让不同顺序变得不同？

不加位置编码时，整体交换 token 的顺序，输出只做相同交换。加入位置编码后，交换 token 内容但保留位置，输出不再只是原输出的交换。

In [ ]:
import numpy as np
def attention(X):
    scores = X@X.T/np.sqrt(X.shape[1]); return softmax(scores)@X
def pe(length, width):
    pos=np.arange(length)[:,None]; dims=np.arange(0,width,2)[None,:]; a=pos/(10000**(dims/width)); out=np.zeros((length,width)); out[:,0::2]=np.sin(a); out[:,1::2]=np.cos(a); return out
X=np.eye(4); order=np.array([2,0,3,1])
error_without=np.max(np.abs(attention(X[order])-attention(X)[order]))
P=pe(4,4); original=attention(X+P); swapped_content=attention(X[order]+P)
error_with=np.max(np.abs(swapped_content-original[order]))
print("不加位置编码的置换误差 =", f"{error_without:.2e}")
print("加位置编码后（位置不动）的差异 =", f"{error_with:.3f}")
assert error_without < 1e-12 and error_with > 1e-3
print("结论：位置编码让“内容在哪里”成为模型可用的信息。")

## 实验 4：注意力的 O(n²) 成本有多快？

注意力分数矩阵有 `n × n` 个元素，所以长度翻倍时，矩阵元素数变成四倍。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
lengths=np.array([128,256,512,1024,2048]); pairs=lengths**2
print("序列从 512 增至 1024：", f"{pairs[3]/pairs[2]:.0f} 倍的注意力分数")
fig, ax=plt.subplots(figsize=(7,4)); ax.plot(lengths,pairs/1e6,"o-",color="#b64926")
ax.set_xscale("log",base=2); ax.set_xticks(lengths,[str(n) for n in lengths]); ax.set_xlabel("序列长度 n"); ax.set_ylabel("注意力分数数量（百万）"); ax.set_title("全连接自注意力的分数矩阵是 n²"); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()